# GSE285335 research evidence audit
Executed September 6, 2026. Run from the project root or research directory using standard-library Python. Checks saved source hashes, donor mapping, and inspection totals. Full expression/QC validation remains future work.


In [1]:
from pathlib import Path
from collections import Counter
import csv, gzip, hashlib, io, json

candidate = Path.cwd()
research = candidate if (candidate / "GSE285335_sample_manifest.tsv").exists() else candidate / "research"
rows = list(csv.DictReader((research / "GSE285335_sample_manifest.tsv").open(), delimiter="\t"))
summary = json.loads((research / "GSE285335_source_summary.json").read_text())
assert len(rows) == 26 == len({r["sample_id"] for r in rows})
assert Counter(r["group"] for r in rows) == {"Early": 11, "Late": 6, "Healthy": 9}
assert sum(int(r["supplied_barcodes"]) for r in rows) == 282463
assert sum(int(r[k+"_compressed_bytes"]) for r in rows for k in ("matrix", "features", "barcodes")) == 1586505246
for filename, hash_key in [("GSE285335_series_matrix.txt.gz", "metadata_sha256"), ("GSE285335_filelist.txt", "filelist_sha256")]:
    assert hashlib.sha256((research / filename).read_bytes()).hexdigest() == summary[hash_key]
assert len({r["feature_content_sha256"] for r in rows}) == 1
assert all(int(r["feature_rows"]) == int(r["unique_feature_ids"]) == 36601 for r in rows)
source = list(csv.reader(io.StringIO(gzip.decompress((research / "GSE285335_series_matrix.txt.gz").read_bytes()).decode()), delimiter="\t"))
ids = next(r[1:] for r in source if r and r[0] == "!Sample_geo_accession")
groups = next(r[1:] for r in source if r and r[0] == "!Sample_characteristics_ch1" and r[1].startswith("disease state:"))
batches = next(r[1:] for r in source if r and r[0] == "!Sample_characteristics_ch1" and r[1].startswith("batch:"))
assert len(ids) == len(groups) == len(batches) == 26
for row, accession, group, batch in zip(rows, ids, groups, batches):
    assert (row["sample_id"], row["group"], row["batch"]) == (accession, group.split(": ", 1)[1], batch.split(": ", 1)[1])
audit = {
    "status": "PASS: saved research evidence is internally consistent",
    "donors": len(rows), "supplied_barcodes": 282463,
    "compressed_expression_bytes": 1586505246,
    "groups": dict(Counter(r["group"] for r in rows)),
    "group_by_batch": dict(Counter(r["group"]+"/"+r["batch"] for r in rows)),
    "source_hashes_match": True, "metadata_alignment_matches": True,
    "limits": "This audits saved metadata and inspection summaries. It does not validate all matrix values or execute the planned Nextflow workflow."
}
print(json.dumps(audit, indent=2))


{
  "status": "PASS: saved research evidence is internally consistent",
  "donors": 26,
  "supplied_barcodes": 282463,
  "compressed_expression_bytes": 1586505246,
  "groups": {
    "Late": 6,
    "Early": 11,
    "Healthy": 9
  },
  "group_by_batch": {
    "Late/First": 2,
    "Late/Second": 4,
    "Early/First": 2,
    "Early/Second": 9,
    "Healthy/First": 4,
    "Healthy/Second": 5
  },
  "source_hashes_match": true,
  "metadata_alignment_matches": true,
  "limits": "This audits saved metadata and inspection summaries. It does not validate all matrix values or execute the planned Nextflow workflow."
}
